# 02 — Pré-processamento e Criação dos Datasets de Modelagem

Este notebook realiza o pré-processamento oficial da base meteorológica de São Paulo e cria os datasets finais para modelagem preditiva.

A base Parquet utilizada aqui **já está filtrada para o estado de São Paulo**.

Objetivos do notebook:

- limpar e padronizar os dados meteorológicos;
- tratar valores sentinela e valores fisicamente inválidos;
- preservar eventos climáticos extremos reais;
- criar variáveis temporais e geográficas;
- gerar uma base horária limpa;
- gerar uma base diária por estação;
- criar o dataset para prever a temperatura de amanhã;
- criar o dataset para prever a temperatura média dos próximos 7 dias;
- separar treino e teste temporalmente;
- aplicar imputação sem vazamento de dados;
- salvar todas as bases em Parquet.

Modelos esperados nas próximas etapas:

- Linear Regression;
- Random Forest Regressor;
- Redes Neurais.

## 1. Inicialização da SparkSession

A `SparkSession` é o ponto de entrada para trabalhar com Spark.

Como o projeto usa um dataset meteorológico grande, o processamento será feito com PySpark e consultas `spark.sql`.

In [41]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Preprocessamento_Weather_SP_Regressao")
    .getOrCreate()
)

spark

## 2. Importação das bibliotecas

As transformações principais usam PySpark.

Este notebook não usa Pandas. Quando necessário, as bases são agregadas no Spark.

In [42]:
from pyspark.sql.functions import (
    col,
    when,
    count,
    regexp_replace,
    regexp_extract,
    to_date,
    year,
    month,
    dayofmonth,
    hour,
    coalesce,
    sin,
    cos,
    lit
)

from pyspark.ml.feature import Imputer, StringIndexer

import unicodedata
import re
import math

## 3. Carregamento da base em Parquet

A base utilizada neste notebook vem do arquivo Parquet gerado no notebook anterior.

O formato Parquet é recomendado em projetos de Big Data porque armazena os dados em formato colunar, o que melhora a performance de leitura, compressão e seleção de colunas.

In [43]:
input_path = "/home/jovyan/work/data/processed/weather_sp_parquet"

df_raw = spark.read.parquet(input_path)

df_raw.createOrReplaceTempView("weather_raw")

spark.sql("""
    SELECT *
    FROM weather_raw
    LIMIT 5
""").show(truncate=False)

df_raw.printSchema()

+------+----------+-------------------+--------------------------------+-----------------------------------------------------+-----------------------------------------------+------------------------------------------------+-----------------------+--------------------------------------------+------------------------------------+------------------------------------------+------------------------------------------+------------------------------------------------+------------------------------------------------+----------------------------------------+----------------------------------------+-----------------------------------+------------------------------------+--------------------------+-------------------------------+------+-----+--------+------------+------------+------------+------+
|index |Data      |Hora               |PRECIPITAÇÃO TOTAL, HORÁRIO (mm)|PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)|PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)|PRESSÃO ATMOSFERICA MIN. NA HO

## 4. Normalização dos nomes das colunas

Os nomes originais das colunas podem conter acentos, espaços, parênteses, vírgulas e outros caracteres especiais.

Para facilitar o uso das colunas em expressões Spark, todos os nomes são padronizados para:

- letras minúsculas;
- sem acentos;
- sem caracteres especiais;
- separação por `_`.

In [44]:
def normalizar_nome_coluna(nome):
    nome = nome.strip()
    nome = unicodedata.normalize("NFKD", nome)
    nome = nome.encode("ASCII", "ignore").decode("utf-8")
    nome = nome.lower()
    nome = re.sub(r"[^a-z0-9]+", "_", nome)
    nome = re.sub(r"_+", "_", nome)
    nome = nome.strip("_")
    return nome

colunas_normalizadas = [normalizar_nome_coluna(c) for c in df_raw.columns]

df = df_raw.toDF(*colunas_normalizadas)

df.columns

['index',
 'data',
 'hora',
 'precipitacao_total_horario_mm',
 'pressao_atmosferica_ao_nivel_da_estacao_horaria_mb',
 'pressao_atmosferica_max_na_hora_ant_aut_mb',
 'pressao_atmosferica_min_na_hora_ant_aut_mb',
 'radiacao_global_kj_m2',
 'temperatura_do_ar_bulbo_seco_horaria_c',
 'temperatura_do_ponto_de_orvalho_c',
 'temperatura_maxima_na_hora_ant_aut_c',
 'temperatura_minima_na_hora_ant_aut_c',
 'temperatura_orvalho_max_na_hora_ant_aut_c',
 'temperatura_orvalho_min_na_hora_ant_aut_c',
 'umidade_rel_max_na_hora_ant_aut',
 'umidade_rel_min_na_hora_ant_aut',
 'umidade_relativa_do_ar_horaria',
 'vento_direcao_horaria_gr_gr',
 'vento_rajada_maxima_m_s',
 'vento_velocidade_horaria_m_s',
 'region',
 'state',
 'station',
 'station_code',
 'latitude',
 'longitude',
 'height']

## 5. Renomeação das variáveis principais

Após a normalização automática, algumas colunas ainda ficam com nomes muito longos.

Nesta etapa, as principais variáveis meteorológicas são renomeadas para nomes mais simples e legíveis. Isso melhora a leitura do código e facilita a etapa posterior de modelagem.

In [45]:
mapa_renomeacao = {
    "precipitacao_total_horario_mm": "precipitacao",
    "pressao_atmosferica_ao_nivel_da_estacao_horaria_mb": "pressao",
    "pressao_atmosferica_max_na_hora_ant_aut_mb": "pressao_maxima",
    "pressao_atmosferica_min_na_hora_ant_aut_mb": "pressao_minima",
    "radiacao_global_kj_m2": "radiacao",
    "temperatura_do_ar_bulbo_seco_horaria_c": "temperatura",
    "temperatura_do_ponto_de_orvalho_c": "temperatura_orvalho",
    "temperatura_maxima_na_hora_ant_aut_c": "temperatura_maxima",
    "temperatura_minima_na_hora_ant_aut_c": "temperatura_minima",
    "temperatura_orvalho_max_na_hora_ant_aut_c": "temperatura_orvalho_maxima",
    "temperatura_orvalho_min_na_hora_ant_aut_c": "temperatura_orvalho_minima",
    "umidade_relativa_do_ar_horaria": "umidade",
    "umidade_rel_max_na_hora_ant_aut": "umidade_maxima",
    "umidade_rel_min_na_hora_ant_aut": "umidade_minima",
    "vento_direcao_horaria_gr_gr": "direcao_vento",
    "vento_rajada_maxima_m_s": "rajada_vento",
    "vento_velocidade_horaria_m_s": "velocidade_vento",
    "height": "altitude"
}

for coluna_antiga, coluna_nova in mapa_renomeacao.items():
    if coluna_antiga in df.columns:
        df = df.withColumnRenamed(coluna_antiga, coluna_nova)

df.printSchema()

root
 |-- index: integer (nullable = true)
 |-- data: date (nullable = true)
 |-- hora: timestamp (nullable = true)
 |-- precipitacao: double (nullable = true)
 |-- pressao: double (nullable = true)
 |-- pressao_maxima: double (nullable = true)
 |-- pressao_minima: double (nullable = true)
 |-- radiacao: integer (nullable = true)
 |-- temperatura: double (nullable = true)
 |-- temperatura_orvalho: double (nullable = true)
 |-- temperatura_maxima: double (nullable = true)
 |-- temperatura_minima: double (nullable = true)
 |-- temperatura_orvalho_maxima: double (nullable = true)
 |-- temperatura_orvalho_minima: double (nullable = true)
 |-- umidade_maxima: integer (nullable = true)
 |-- umidade_minima: integer (nullable = true)
 |-- umidade: integer (nullable = true)
 |-- direcao_vento: integer (nullable = true)
 |-- rajada_vento: double (nullable = true)
 |-- velocidade_vento: double (nullable = true)
 |-- region: string (nullable = true)
 |-- state: string (nullable = true)
 |-- statio

## 6. Seleção das colunas relevantes

Como o Parquet já representa São Paulo, o foco é manter:

- variáveis temporais;
- variáveis geográficas;
- identificadores de estação;
- variáveis meteorológicas.

In [46]:
df.createOrReplaceTempView("weather_sp_renomeado")

In [47]:
df = spark.sql("""
    SELECT
        data,
        hora,
        region,
        state,
        station,
        station_code,
        latitude,
        longitude,
        altitude,
        temperatura,
        temperatura_maxima,
        temperatura_minima,
        temperatura_orvalho,
        temperatura_orvalho_maxima,
        temperatura_orvalho_minima,
        umidade,
        umidade_maxima,
        umidade_minima,
        pressao,
        pressao_maxima,
        pressao_minima,
        precipitacao,
        radiacao,
        velocidade_vento,
        rajada_vento,
        direcao_vento
    FROM weather_sp_renomeado
""")

df.createOrReplaceTempView("weather_sp_selecionado")

df.show(5, truncate=False)
df.printSchema()

+----------+-------------------+------+-----+--------+------------+------------+------------+--------+-----------+------------------+------------------+-------------------+--------------------------+--------------------------+-------+--------------+--------------+-------+--------------+--------------+------------+--------+----------------+------------+-------------+
|data      |hora               |region|state|station |station_code|latitude    |longitude   |altitude|temperatura|temperatura_maxima|temperatura_minima|temperatura_orvalho|temperatura_orvalho_maxima|temperatura_orvalho_minima|umidade|umidade_maxima|umidade_minima|pressao|pressao_maxima|pressao_minima|precipitacao|radiacao|velocidade_vento|rajada_vento|direcao_vento|
+----------+-------------------+------+-----+--------+------------+------------+------------+--------+-----------+------------------+------------------+-------------------+--------------------------+--------------------------+-------+--------------+-------------

## 7. Conversão das colunas numéricas

Nesta etapa, as variáveis meteorológicas e geográficas são convertidas para o tipo `double` usando `spark.sql`.

Também é feita a troca de vírgula por ponto em formato textual, garantindo compatibilidade com o separador decimal esperado pelo Spark.

In [48]:
df.createOrReplaceTempView("weather_sp_selecionado")

df = spark.sql("""
    SELECT
        data,
        hora,
        region,
        state,
        station,
        station_code,

        CAST(REPLACE(CAST(latitude AS STRING), ',', '.') AS DOUBLE) AS latitude,
        CAST(REPLACE(CAST(longitude AS STRING), ',', '.') AS DOUBLE) AS longitude,
        CAST(REPLACE(CAST(altitude AS STRING), ',', '.') AS DOUBLE) AS altitude,

        CAST(REPLACE(CAST(temperatura AS STRING), ',', '.') AS DOUBLE) AS temperatura,
        CAST(REPLACE(CAST(temperatura_maxima AS STRING), ',', '.') AS DOUBLE) AS temperatura_maxima,
        CAST(REPLACE(CAST(temperatura_minima AS STRING), ',', '.') AS DOUBLE) AS temperatura_minima,
        CAST(REPLACE(CAST(temperatura_orvalho AS STRING), ',', '.') AS DOUBLE) AS temperatura_orvalho,
        CAST(REPLACE(CAST(temperatura_orvalho_maxima AS STRING), ',', '.') AS DOUBLE) AS temperatura_orvalho_maxima,
        CAST(REPLACE(CAST(temperatura_orvalho_minima AS STRING), ',', '.') AS DOUBLE) AS temperatura_orvalho_minima,

        CAST(REPLACE(CAST(umidade AS STRING), ',', '.') AS DOUBLE) AS umidade,
        CAST(REPLACE(CAST(umidade_maxima AS STRING), ',', '.') AS DOUBLE) AS umidade_maxima,
        CAST(REPLACE(CAST(umidade_minima AS STRING), ',', '.') AS DOUBLE) AS umidade_minima,

        CAST(REPLACE(CAST(pressao AS STRING), ',', '.') AS DOUBLE) AS pressao,
        CAST(REPLACE(CAST(pressao_maxima AS STRING), ',', '.') AS DOUBLE) AS pressao_maxima,
        CAST(REPLACE(CAST(pressao_minima AS STRING), ',', '.') AS DOUBLE) AS pressao_minima,

        CAST(REPLACE(CAST(precipitacao AS STRING), ',', '.') AS DOUBLE) AS precipitacao,
        CAST(REPLACE(CAST(radiacao AS STRING), ',', '.') AS DOUBLE) AS radiacao,
        CAST(REPLACE(CAST(velocidade_vento AS STRING), ',', '.') AS DOUBLE) AS velocidade_vento,
        CAST(REPLACE(CAST(rajada_vento AS STRING), ',', '.') AS DOUBLE) AS rajada_vento,
        CAST(REPLACE(CAST(direcao_vento AS STRING), ',', '.') AS DOUBLE) AS direcao_vento

    FROM weather_sp_selecionado
""")

df.createOrReplaceTempView("weather_sp_numerico")

df.printSchema()

root
 |-- data: date (nullable = true)
 |-- hora: timestamp (nullable = true)
 |-- region: string (nullable = true)
 |-- state: string (nullable = true)
 |-- station: string (nullable = true)
 |-- station_code: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- altitude: double (nullable = true)
 |-- temperatura: double (nullable = true)
 |-- temperatura_maxima: double (nullable = true)
 |-- temperatura_minima: double (nullable = true)
 |-- temperatura_orvalho: double (nullable = true)
 |-- temperatura_orvalho_maxima: double (nullable = true)
 |-- temperatura_orvalho_minima: double (nullable = true)
 |-- umidade: double (nullable = true)
 |-- umidade_maxima: double (nullable = true)
 |-- umidade_minima: double (nullable = true)
 |-- pressao: double (nullable = true)
 |-- pressao_maxima: double (nullable = true)
 |-- pressao_minima: double (nullable = true)
 |-- precipitacao: double (nullable = true)
 |-- radiacao: double (null

## 8. Tratamento de valores sentinela

No dataset, valores como `-9999`, `-9999.0` e `-999` não representam medições reais.  
Eles são códigos sentinela usados para indicar ausência de informação.

Nesta etapa, esses valores são convertidos para `NULL` usando `spark.sql`.

A partir daqui, os valores ausentes passam a ser tratados como nulos reais pelo Spark.

In [49]:
df.createOrReplaceTempView("weather_sp_numerico")

df = spark.sql("""
    SELECT
        data,
        hora,
        region,
        state,
        station,
        station_code,

        latitude,
        longitude,
        altitude,

        CASE WHEN temperatura <= -999 THEN NULL ELSE temperatura END AS temperatura,
        CASE WHEN temperatura_maxima <= -999 THEN NULL ELSE temperatura_maxima END AS temperatura_maxima,
        CASE WHEN temperatura_minima <= -999 THEN NULL ELSE temperatura_minima END AS temperatura_minima,
        CASE WHEN temperatura_orvalho <= -999 THEN NULL ELSE temperatura_orvalho END AS temperatura_orvalho,
        CASE WHEN temperatura_orvalho_maxima <= -999 THEN NULL ELSE temperatura_orvalho_maxima END AS temperatura_orvalho_maxima,
        CASE WHEN temperatura_orvalho_minima <= -999 THEN NULL ELSE temperatura_orvalho_minima END AS temperatura_orvalho_minima,

        CASE WHEN umidade <= -999 THEN NULL ELSE umidade END AS umidade,
        CASE WHEN umidade_maxima <= -999 THEN NULL ELSE umidade_maxima END AS umidade_maxima,
        CASE WHEN umidade_minima <= -999 THEN NULL ELSE umidade_minima END AS umidade_minima,

        CASE WHEN pressao <= -999 THEN NULL ELSE pressao END AS pressao,
        CASE WHEN pressao_maxima <= -999 THEN NULL ELSE pressao_maxima END AS pressao_maxima,
        CASE WHEN pressao_minima <= -999 THEN NULL ELSE pressao_minima END AS pressao_minima,

        CASE WHEN precipitacao <= -999 THEN NULL ELSE precipitacao END AS precipitacao,
        CASE WHEN radiacao <= -999 THEN NULL ELSE radiacao END AS radiacao,
        CASE WHEN velocidade_vento <= -999 THEN NULL ELSE velocidade_vento END AS velocidade_vento,
        CASE WHEN rajada_vento <= -999 THEN NULL ELSE rajada_vento END AS rajada_vento,
        CASE WHEN direcao_vento <= -999 THEN NULL ELSE direcao_vento END AS direcao_vento

    FROM weather_sp_numerico
""")

df.createOrReplaceTempView("weather_sp_sem_sentinelas")

df.show(5, truncate=False)

+----------+-------------------+------+-----+--------+------------+------------+------------+--------+-----------+------------------+------------------+-------------------+--------------------------+--------------------------+-------+--------------+--------------+-------+--------------+--------------+------------+--------+----------------+------------+-------------+
|data      |hora               |region|state|station |station_code|latitude    |longitude   |altitude|temperatura|temperatura_maxima|temperatura_minima|temperatura_orvalho|temperatura_orvalho_maxima|temperatura_orvalho_minima|umidade|umidade_maxima|umidade_minima|pressao|pressao_maxima|pressao_minima|precipitacao|radiacao|velocidade_vento|rajada_vento|direcao_vento|
+----------+-------------------+------+-----+--------+------------+------------+------------+--------+-----------+------------------+------------------+-------------------+--------------------------+--------------------------+-------+--------------+-------------

## 9. Criação das variáveis temporais

Nesta etapa, a data e a hora são transformadas em variáveis úteis para análise temporal e modelagem.

A coluna `data_formatada` padroniza a data, enquanto `ano`, `mes`, `dia` e `hora_num` permitem agregações por período.

Como a coluna `hora` pode aparecer como timestamp, o código usa `HOUR(hora)` e mantém uma alternativa com expressão regular para casos em que a hora esteja em formato textual.

In [50]:
df.createOrReplaceTempView("weather_sp_sem_sentinelas")

df = spark.sql("""
    SELECT
        data,
        hora,
        region,
        state,
        station,
        station_code,

        latitude,
        longitude,
        altitude,

        temperatura,
        temperatura_maxima,
        temperatura_minima,
        temperatura_orvalho,
        temperatura_orvalho_maxima,
        temperatura_orvalho_minima,

        umidade,
        umidade_maxima,
        umidade_minima,

        pressao,
        pressao_maxima,
        pressao_minima,

        precipitacao,
        radiacao,
        velocidade_vento,
        rajada_vento,
        direcao_vento,

        COALESCE(
            TO_DATE(data, 'yyyy-MM-dd'),
            TO_DATE(data, 'dd/MM/yyyy')
        ) AS data_formatada,

        YEAR(
            COALESCE(
                TO_DATE(data, 'yyyy-MM-dd'),
                TO_DATE(data, 'dd/MM/yyyy')
            )
        ) AS ano,

        MONTH(
            COALESCE(
                TO_DATE(data, 'yyyy-MM-dd'),
                TO_DATE(data, 'dd/MM/yyyy')
            )
        ) AS mes,

        DAY(
            COALESCE(
                TO_DATE(data, 'yyyy-MM-dd'),
                TO_DATE(data, 'dd/MM/yyyy')
            )
        ) AS dia,

        COALESCE(
            HOUR(hora),
            CAST(REGEXP_EXTRACT(CAST(hora AS STRING), '(\\\\d{2})', 1) AS INT)
        ) AS hora_num

    FROM weather_sp_sem_sentinelas
""")

df.createOrReplaceTempView("weather_sp_temporal")

spark.sql("""
    SELECT
        data,
        data_formatada,
        ano,
        mes,
        dia,
        hora,
        hora_num
    FROM weather_sp_temporal
    LIMIT 20
""").show(truncate=False)

+----------+--------------+----+---+---+-------------------+--------+
|data      |data_formatada|ano |mes|dia|hora               |hora_num|
+----------+--------------+----+---+---+-------------------+--------+
|2019-02-16|2019-02-16    |2019|2  |16 |2026-05-10 19:00:00|19      |
|2019-02-16|2019-02-16    |2019|2  |16 |2026-05-10 20:00:00|20      |
|2019-02-16|2019-02-16    |2019|2  |16 |2026-05-10 21:00:00|21      |
|2019-02-16|2019-02-16    |2019|2  |16 |2026-05-10 22:00:00|22      |
|2019-02-16|2019-02-16    |2019|2  |16 |2026-05-10 23:00:00|23      |
|2019-02-17|2019-02-17    |2019|2  |17 |2026-05-10 00:00:00|0       |
|2019-02-17|2019-02-17    |2019|2  |17 |2026-05-10 01:00:00|1       |
|2019-02-17|2019-02-17    |2019|2  |17 |2026-05-10 02:00:00|2       |
|2019-02-17|2019-02-17    |2019|2  |17 |2026-05-10 03:00:00|3       |
|2019-02-17|2019-02-17    |2019|2  |17 |2026-05-10 04:00:00|4       |
|2019-02-17|2019-02-17    |2019|2  |17 |2026-05-10 05:00:00|5       |
|2019-02-17|2019-02-

## 10. Criação de variáveis temporais cíclicas

Mês e hora possuem comportamento cíclico.  
Por exemplo, dezembro e janeiro estão próximos no calendário, assim como 23h e 0h estão próximas no ciclo diário.

Nesta etapa, são criadas variáveis com seno e cosseno usando `spark.sql`, permitindo que os modelos capturem melhor padrões sazonais e horários.

In [51]:
df.createOrReplaceTempView("weather_sp_temporal")

df = spark.sql("""
    SELECT
        data,
        hora,
        region,
        state,
        station,
        station_code,
        latitude,
        longitude,
        altitude,
        temperatura,
        temperatura_maxima,
        temperatura_minima,
        temperatura_orvalho,
        temperatura_orvalho_maxima,
        temperatura_orvalho_minima,
        umidade,
        umidade_maxima,
        umidade_minima,
        pressao,
        pressao_maxima,
        pressao_minima,
        precipitacao,
        radiacao,
        velocidade_vento,
        rajada_vento,
        direcao_vento,
        data_formatada,
        ano,
        mes,
        dia,
        hora_num,

        SIN(2 * PI() * mes / 12) AS mes_sin,
        COS(2 * PI() * mes / 12) AS mes_cos,
        SIN(2 * PI() * hora_num / 24) AS hora_sin,
        COS(2 * PI() * hora_num / 24) AS hora_cos

    FROM weather_sp_temporal
""")

df.createOrReplaceTempView("weather_sp_temporal_ciclico")

spark.sql("""
    SELECT
        mes,
        mes_sin,
        mes_cos,
        hora_num,
        hora_sin,
        hora_cos
    FROM weather_sp_temporal_ciclico
    LIMIT 20
""").show(truncate=False)

+---+------------------+------------------+--------+----------------------+---------------------+
|mes|mes_sin           |mes_cos           |hora_num|hora_sin              |hora_cos             |
+---+------------------+------------------+--------+----------------------+---------------------+
|2  |0.8660254037844386|0.5000000000000001|19      |-0.9659258262890684   |0.2588190451025203   |
|2  |0.8660254037844386|0.5000000000000001|20      |-0.8660254037844386   |0.5000000000000001   |
|2  |0.8660254037844386|0.5000000000000001|21      |-0.7071067811865477   |0.7071067811865474   |
|2  |0.8660254037844386|0.5000000000000001|22      |-0.5000000000000004   |0.8660254037844384   |
|2  |0.8660254037844386|0.5000000000000001|23      |-0.25881904510252157  |0.9659258262890681   |
|2  |0.8660254037844386|0.5000000000000001|0       |0.0                   |1.0                  |
|2  |0.8660254037844386|0.5000000000000001|1       |0.25881904510252074   |0.9659258262890683   |
|2  |0.8660254037844

## 11. Confirmação do recorte de São Paulo

Como o arquivo Parquet utilizado neste notebook já está filtrado para o estado de São Paulo, esta etapa apenas confirma o conteúdo da base.

A consulta mostra o estado presente, a quantidade total de registros, o número de estações meteorológicas e o período disponível.

In [52]:
df.createOrReplaceTempView("weather_sp_temporal_ciclico")

spark.sql("""
    SELECT
        state,
        COUNT(*) AS total_registros,
        COUNT(DISTINCT station_code) AS total_estacoes,
        MIN(data_formatada) AS data_inicial,
        MAX(data_formatada) AS data_final
    FROM weather_sp_temporal_ciclico
    GROUP BY state
    ORDER BY state
""").show(truncate=False)

+-----+---------------+--------------+------------+----------+
|state|total_registros|total_estacoes|data_inicial|data_final|
+-----+---------------+--------------+------------+----------+
|SP   |4288560        |43            |2001-08-30  |2021-04-30|
+-----+---------------+--------------+------------+----------+



## 12. Remoção de duplicatas

Como os dados são horários, espera-se uma medição por estação, data e hora.

In [53]:
df.createOrReplaceTempView("weather_sp_temporal_ciclico")

total_antes_duplicatas = spark.sql("""
    SELECT COUNT(*) AS total
    FROM weather_sp_temporal_ciclico
""").collect()[0]["total"]

df = spark.sql("""
    SELECT *
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY station_code, data_formatada, hora_num
                ORDER BY data_formatada
            ) AS numero_linha
        FROM weather_sp_temporal_ciclico
    ) tabela_com_rn
    WHERE numero_linha = 1
""")

df.createOrReplaceTempView("weather_sp_sem_duplicatas")

coords_por_estacao = spark.sql("""
    SELECT
        station_code,
        FIRST(station, true) AS station_nome_canonical,
        ROUND(AVG(latitude), 4) AS latitude_media,
        ROUND(AVG(longitude), 4) AS longitude_media,
        ROUND(AVG(altitude), 2) AS altitude_media
    FROM weather_sp_sem_duplicatas
    WHERE station_code IS NOT NULL
    GROUP BY station_code
""")

coords_por_estacao.createOrReplaceTempView("coords_por_estacao")

df = spark.sql("""
    SELECT
        w.data,
        w.hora,
        w.region,
        w.state,
        c.station_nome_canonical AS station,
        w.station_code,
        c.latitude_media AS latitude,
        c.longitude_media AS longitude,
        c.altitude_media AS altitude,
        w.temperatura,
        w.temperatura_maxima,
        w.temperatura_minima,
        w.temperatura_orvalho,
        w.temperatura_orvalho_maxima,
        w.temperatura_orvalho_minima,
        w.umidade,
        w.umidade_maxima,
        w.umidade_minima,
        w.pressao,
        w.pressao_maxima,
        w.pressao_minima,
        w.precipitacao,
        w.radiacao,
        w.velocidade_vento,
        w.rajada_vento,
        w.direcao_vento,
        w.data_formatada,
        w.ano,
        w.mes,
        w.dia,
        w.hora_num,
        w.mes_sin,
        w.mes_cos,
        w.hora_sin,
        w.hora_cos
    FROM weather_sp_sem_duplicatas w
    JOIN coords_por_estacao c ON w.station_code = c.station_code
""")

df.createOrReplaceTempView("weather_sp_sem_duplicatas")

total_depois_duplicatas = spark.sql("""
    SELECT COUNT(*) AS total
    FROM weather_sp_sem_duplicatas
""").collect()[0]["total"]

print(f"Linhas antes: {total_antes_duplicatas:,}")
print(f"Linhas depois: {total_depois_duplicatas:,}")
print(f"Duplicatas removidas: {total_antes_duplicatas - total_depois_duplicatas:,}")

Linhas antes: 4,288,560
Linhas depois: 4,288,560
Duplicatas removidas: 0


## 13. Aplicação de regras físicas

Nesta etapa são removidos apenas valores fisicamente inválidos, usando `spark.sql`.

Valores nulos são preservados, pois representam medições ausentes que serão tratadas posteriormente por imputação quando forem usadas como features.

Não será aplicada remoção genérica por IQR, porque eventos climáticos extremos podem representar fenômenos reais, como ondas de calor, frentes frias ou períodos de chuva intensa.

In [54]:
total_antes = spark.sql("""
    SELECT COUNT(*) AS total
    FROM weather_sp_sem_duplicatas
""").collect()[0]["total"]

df_tratado = spark.sql("""
    SELECT *
    FROM weather_sp_sem_duplicatas
    WHERE
        (temperatura IS NULL OR temperatura BETWEEN -10 AND 50)

        AND (temperatura_maxima IS NULL OR temperatura_maxima BETWEEN -30 AND 50)
        AND (temperatura_minima IS NULL OR temperatura_minima BETWEEN -30 AND 50)
        AND (temperatura_orvalho IS NULL OR temperatura_orvalho BETWEEN -30 AND 50)
        AND (temperatura_orvalho_maxima IS NULL OR temperatura_orvalho_maxima BETWEEN -30 AND 50)
        AND (temperatura_orvalho_minima IS NULL OR temperatura_orvalho_minima BETWEEN -30 AND 50)

        AND (umidade IS NULL OR umidade BETWEEN 0 AND 100)
        AND (umidade_maxima IS NULL OR umidade_maxima BETWEEN 0 AND 100)
        AND (umidade_minima IS NULL OR umidade_minima BETWEEN 0 AND 100)

        AND (pressao IS NULL OR pressao BETWEEN 800 AND 1100)
        AND (pressao_maxima IS NULL OR pressao_maxima BETWEEN 800 AND 1100)
        AND (pressao_minima IS NULL OR pressao_minima BETWEEN 800 AND 1100)

        AND (precipitacao IS NULL OR precipitacao >= 0)
        AND (radiacao IS NULL OR radiacao >= 0)
        AND (velocidade_vento IS NULL OR velocidade_vento >= 0)
        AND (rajada_vento IS NULL OR rajada_vento >= 0)

        AND (direcao_vento IS NULL OR direcao_vento BETWEEN 0 AND 360)
""")

df_tratado.createOrReplaceTempView("weather_sp_regras_fisicas")

total_depois = spark.sql("""
    SELECT COUNT(*) AS total
    FROM weather_sp_regras_fisicas
""").collect()[0]["total"]

print(f"Antes das regras físicas: {total_antes:,}")
print(f"Após regras físicas     : {total_depois:,}")
print(f"Linhas removidas        : {total_antes - total_depois:,}")

Antes das regras físicas: 4,288,560
Após regras físicas     : 4,278,809
Linhas removidas        : 9,751


In [55]:
spark.sql("SHOW VIEWS").show(truncate=False)

+---------+-----------------------------+-----------+
|namespace|viewName                     |isTemporary|
+---------+-----------------------------+-----------+
|         |base_diaria_features         |true       |
|         |base_diaria_sp               |true       |
|         |coords_por_estacao           |true       |
|         |dataset_amanha               |true       |
|         |dataset_amanha_com_null      |true       |
|         |dataset_amanha_final         |true       |
|         |dataset_semana               |true       |
|         |dataset_semana_com_null      |true       |
|         |dataset_semana_final         |true       |
|         |df_amanha_base               |true       |
|         |df_semana_base               |true       |
|         |weather_raw                  |true       |
|         |weather_sp_com_numero_linha  |true       |
|         |weather_sp_filtros_essenciais|true       |
|         |weather_sp_numerico          |true       |
|         |weather_sp_regras

## 14. Remoção de registros sem temperatura e sem tempo válido

Como o objetivo do projeto é prever temperatura, registros sem valor de temperatura não podem ser usados para criar os alvos supervisionados.

Também são removidos registros sem data, ano, mês ou hora válidos, pois essas informações são necessárias para construir a base diária, as janelas temporais e os alvos futuros.

Esta etapa é feita com `spark.sql`.

In [56]:
total_antes_filtros = spark.sql("""
    SELECT COUNT(*) AS total
    FROM weather_sp_regras_fisicas
""").collect()[0]["total"]

df_tratado = spark.sql("""
    SELECT *
    FROM weather_sp_regras_fisicas
    WHERE temperatura IS NOT NULL
      AND data_formatada IS NOT NULL
      AND ano IS NOT NULL
      AND mes IS NOT NULL
      AND hora_num IS NOT NULL
""")

df_tratado.createOrReplaceTempView("weather_sp_filtros_essenciais")

total_depois_filtros = spark.sql("""
    SELECT COUNT(*) AS total
    FROM weather_sp_filtros_essenciais
""").collect()[0]["total"]

print(f"Antes dos filtros essenciais: {total_antes_filtros:,}")
print(f"Após filtros essenciais     : {total_depois_filtros:,}")
print(f"Linhas removidas            : {total_antes_filtros - total_depois_filtros:,}")

Antes dos filtros essenciais: 4,278,809
Após filtros essenciais     : 3,849,112
Linhas removidas            : 429,697


## 15. Classificações geográficas

Nesta etapa são criadas classificações geográficas aproximadas para enriquecer a análise e a modelagem.

As novas colunas são:

- `faixa_altitude`: classifica as estações em baixa, média ou alta altitude;
- `macro_regiao_sp`: classifica a posição aproximada da estação dentro do estado de São Paulo;
- `tipo_area`: diferencia litoral, serra/altitude, urbano/metropolitano e interior.

Essas classificações são criadas com `spark.sql` usando expressões `CASE WHEN`.

Elas não substituem uma divisão geográfica oficial, mas ajudam a capturar diferenças climáticas relevantes dentro do estado.

In [57]:
df_tratado = spark.sql("""
    SELECT
        *,

        CASE
            WHEN altitude IS NULL THEN 'sem_info'
            WHEN altitude < 300 THEN 'baixa_altitude'
            WHEN altitude < 700 THEN 'media_altitude'
            ELSE 'alta_altitude'
        END AS faixa_altitude,

        CASE
            WHEN longitude <= -48.5 AND latitude <= -23.5 THEN 'sul_sudoeste'
            WHEN longitude <= -48.5 AND latitude > -23.5 THEN 'oeste_noroeste'
            WHEN longitude > -47.0 AND latitude <= -23.5 THEN 'leste_sudeste'
            WHEN longitude > -47.0 AND latitude > -23.5 THEN 'nordeste_vale'
            ELSE 'centro_metropolitana'
        END AS macro_regiao_sp,

        CASE
            WHEN altitude < 100 AND longitude > -47.0 THEN 'litoral'
            WHEN altitude >= 700 THEN 'serra_altitude'
            WHEN station RLIKE '(?i)SAO PAULO|SÃO PAULO|BARUERI|GUARULHOS|OSASCO|SANTO ANDRE|SANTO ANDRÉ|SAO BERNARDO|SÃO BERNARDO'
                THEN 'urbano_metropolitano'
            ELSE 'interior'
        END AS tipo_area

    FROM weather_sp_filtros_essenciais
""")

df_tratado.createOrReplaceTempView("weather_sp_tratado")

spark.sql("""
    SELECT
        macro_regiao_sp,
        tipo_area,
        faixa_altitude,
        COUNT(*) AS total_registros,
        COUNT(DISTINCT station_code) AS total_estacoes
    FROM weather_sp_tratado
    GROUP BY macro_regiao_sp, tipo_area, faixa_altitude
    ORDER BY macro_regiao_sp, tipo_area, faixa_altitude
""").show(100, truncate=False)

+--------------------+--------------+--------------+---------------+--------------+
|macro_regiao_sp     |tipo_area     |faixa_altitude|total_registros|total_estacoes|
+--------------------+--------------+--------------+---------------+--------------+
|centro_metropolitana|interior      |baixa_altitude|146110         |2             |
|centro_metropolitana|interior      |media_altitude|699476         |8             |
|centro_metropolitana|serra_altitude|alta_altitude |370219         |3             |
|leste_sudeste       |litoral       |baixa_altitude|51787          |2             |
|leste_sudeste       |serra_altitude|alta_altitude |139970         |3             |
|nordeste_vale       |interior      |media_altitude|244872         |3             |
|nordeste_vale       |serra_altitude|alta_altitude |368587         |3             |
|oeste_noroeste      |interior      |media_altitude|1597098        |16            |
|oeste_noroeste      |serra_altitude|alta_altitude |108170         |1       

## 16. Percentual de nulos por coluna com Spark SQL

Nesta etapa é calculado o percentual de valores nulos nas principais variáveis meteorológicas e geográficas após o tratamento dos valores sentinela e a aplicação das regras físicas.

Esse resultado ajuda a entender quais variáveis ainda possuem ausência de dados antes da criação das bases de modelagem.

A variável `temperatura` deve estar sem nulos, pois registros sem temperatura foram removidos na etapa anterior. Já variáveis como `radiacao`, `vento` e `precipitacao` podem manter valores ausentes, que serão tratados posteriormente por imputação nas bases de treino e teste.

In [58]:
df_tratado.createOrReplaceTempView("weather_sp_tratado")

spark.sql("""
    SELECT
        ROUND(SUM(CASE WHEN temperatura IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS temperatura,
        ROUND(SUM(CASE WHEN umidade IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS umidade,
        ROUND(SUM(CASE WHEN pressao IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS pressao,
        ROUND(SUM(CASE WHEN precipitacao IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS precipitacao,
        ROUND(SUM(CASE WHEN radiacao IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS radiacao,
        ROUND(SUM(CASE WHEN velocidade_vento IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS velocidade_vento,
        ROUND(SUM(CASE WHEN rajada_vento IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS rajada_vento,
        ROUND(SUM(CASE WHEN direcao_vento IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS direcao_vento,
        ROUND(SUM(CASE WHEN altitude IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS altitude,
        ROUND(SUM(CASE WHEN latitude IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS latitude,
        ROUND(SUM(CASE WHEN longitude IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS longitude
    FROM weather_sp_tratado
""").show(truncate=False)

+-----------+-------+-------+------------+--------+----------------+------------+-------------+--------+--------+---------+
|temperatura|umidade|pressao|precipitacao|radiacao|velocidade_vento|rajada_vento|direcao_vento|altitude|latitude|longitude|
+-----------+-------+-------+------------+--------+----------------+------------+-------------+--------+--------+---------+
|0.0        |1.01   |1.82   |3.74        |44.59   |4.2             |4.27        |4.09         |0.0     |0.0     |0.0      |
+-----------+-------+-------+------------+--------+----------------+------------+-------------+--------+--------+---------+



## 17. Salvamento da base horária limpa

Esta base preserva a granularidade horária e pode ser usada para análises detalhadas.

In [59]:
base_horaria_path = "/home/jovyan/work/data/processed/weather_sp_limpo_horario"

df_tratado.coalesce(4).write.mode("overwrite").parquet(base_horaria_path)

print(f"Base horária limpa salva em: {base_horaria_path}")

Base horária limpa salva em: /home/jovyan/work/data/processed/weather_sp_limpo_horario


## 18. Criação da base diária por estação

Como o objetivo do projeto é prever a temperatura de amanhã e a temperatura média dos próximos 7 dias, a base horária é agregada para uma granularidade diária.

Cada linha da nova base representa uma estação meteorológica em um determinado dia.

Nesta etapa, são calculadas variáveis diárias como:

- temperatura média, mínima e máxima do dia;
- umidade média, mínima e máxima;
- pressão média;
- precipitação total;
- radiação média;
- vento médio;
- rajada máxima.

A agregação é feita com `spark.sql`, agrupando por estação, data e informações geográficas.

In [60]:
base_diaria = spark.sql("""
    SELECT
        station,
        station_code,
        data_formatada,
        ano,
        mes,
        latitude,
        longitude,
        altitude,
        macro_regiao_sp,
        tipo_area,
        faixa_altitude,

        ROUND(AVG(temperatura), 2) AS temp_media_dia,
        ROUND(MIN(temperatura), 2) AS temp_min_dia,
        ROUND(MAX(temperatura), 2) AS temp_max_dia,

        ROUND(AVG(temperatura_orvalho), 2) AS temp_orvalho_media_dia,

        ROUND(AVG(umidade), 2) AS umidade_media_dia,
        ROUND(MIN(umidade), 2) AS umidade_min_dia,
        ROUND(MAX(umidade), 2) AS umidade_max_dia,

        ROUND(AVG(pressao), 2) AS pressao_media_dia,
        ROUND(SUM(precipitacao), 2) AS precipitacao_total_dia,
        ROUND(AVG(radiacao), 2) AS radiacao_media_dia,
        ROUND(AVG(velocidade_vento), 2) AS vento_medio_dia,
        ROUND(MAX(rajada_vento), 2) AS rajada_max_dia,

        COUNT(temperatura) AS medicoes_temp_validas
    FROM weather_sp_tratado
    WHERE data_formatada IS NOT NULL
    GROUP BY
        station,
        station_code,
        data_formatada,
        ano,
        mes,
        latitude,
        longitude,
        altitude,
        macro_regiao_sp,
        tipo_area,
        faixa_altitude
    ORDER BY station_code, data_formatada
""")

base_diaria.createOrReplaceTempView("base_diaria_sp")

base_diaria.show(20, truncate=False)

print(f"Linhas da base diária: {base_diaria.count():,}")

+-------------------+------------+--------------+----+---+--------+---------+--------+---------------+--------------+--------------+--------------+------------+------------+----------------------+-----------------+---------------+---------------+-----------------+----------------------+------------------+---------------+--------------+---------------------+
|station            |station_code|data_formatada|ano |mes|latitude|longitude|altitude|macro_regiao_sp|tipo_area     |faixa_altitude|temp_media_dia|temp_min_dia|temp_max_dia|temp_orvalho_media_dia|umidade_media_dia|umidade_min_dia|umidade_max_dia|pressao_media_dia|precipitacao_total_dia|radiacao_media_dia|vento_medio_dia|rajada_max_dia|medicoes_temp_validas|
+-------------------+------------+--------------+----+---+--------+---------+--------+---------------+--------------+--------------+--------------+------------+------------+----------------------+-----------------+---------------+---------------+-----------------+----------------

## 19. Criação de features temporais diárias

São criadas variáveis cíclicas de mês e estatísticas móveis dos últimos dias.

Essas variáveis serão usadas para prever temperatura futura.

In [61]:
base_diaria.createOrReplaceTempView("base_diaria_sp")

base_diaria_features = spark.sql("""
    SELECT
        *,

        SIN(2 * PI() * mes / 12) AS mes_sin,
        COS(2 * PI() * mes / 12) AS mes_cos,

        LAG(temp_media_dia, 1) OVER (
            PARTITION BY station_code
            ORDER BY data_formatada
        ) AS temp_media_ontem,

        AVG(temp_media_dia) OVER (
            PARTITION BY station_code
            ORDER BY data_formatada
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS temp_media_ultimos_3_dias,

        AVG(temp_media_dia) OVER (
            PARTITION BY station_code
            ORDER BY data_formatada
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS temp_media_ultimos_7_dias,

        AVG(umidade_media_dia) OVER (
            PARTITION BY station_code
            ORDER BY data_formatada
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS umidade_media_ultimos_7_dias,

        SUM(precipitacao_total_dia) OVER (
            PARTITION BY station_code
            ORDER BY data_formatada
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS precipitacao_ultimos_7_dias

    FROM base_diaria_sp
""")

base_diaria_features.createOrReplaceTempView("base_diaria_features")

base_diaria_features.show(20, truncate=False)

+--------+------------+--------------+----+---+--------+---------+--------+--------------------+---------+--------------+--------------+------------+------------+----------------------+-----------------+---------------+---------------+-----------------+----------------------+------------------+---------------+--------------+---------------------+-------------------+-----------------------+----------------+-------------------------+-------------------------+----------------------------+---------------------------+
|station |station_code|data_formatada|ano |mes|latitude|longitude|altitude|macro_regiao_sp     |tipo_area|faixa_altitude|temp_media_dia|temp_min_dia|temp_max_dia|temp_orvalho_media_dia|umidade_media_dia|umidade_min_dia|umidade_max_dia|pressao_media_dia|precipitacao_total_dia|radiacao_media_dia|vento_medio_dia|rajada_max_dia|medicoes_temp_validas|mes_sin            |mes_cos                |temp_media_ontem|temp_media_ultimos_3_dias|temp_media_ultimos_7_dias|umidade_media_ultimo

## 20. Criação do dataset para prever a temperatura de amanhã

O alvo será:

`temperatura_amanha`

Ele representa a temperatura média da mesma estação no dia seguinte, ou seja, para cada estação, ordena os dias pela data e pega a temperatura média do próximo dia.

In [62]:
base_diaria_features.createOrReplaceTempView("base_diaria_features")

dataset_amanha = spark.sql("""
    SELECT
        *,
        LEAD(temp_media_dia, 1) OVER (
            PARTITION BY station_code
            ORDER BY data_formatada
        ) AS temperatura_amanha
    FROM base_diaria_features
""")

dataset_amanha.createOrReplaceTempView("dataset_amanha_com_null")

dataset_amanha = spark.sql("""
    SELECT *
    FROM dataset_amanha_com_null
    WHERE temperatura_amanha IS NOT NULL
""")

dataset_amanha.createOrReplaceTempView("dataset_amanha")

spark.sql("""
    SELECT
        COUNT(*) AS total_linhas,
        MIN(data_formatada) AS data_inicial,
        MAX(data_formatada) AS data_final,
        ROUND(AVG(temperatura_amanha), 2) AS media_alvo
    FROM dataset_amanha
""").show(truncate=False)

dataset_amanha.show(10, truncate=False)

+------------+------------+----------+----------+
|total_linhas|data_inicial|data_final|media_alvo|
+------------+------------+----------+----------+
|166172      |2001-08-30  |2021-04-29|21.74     |
+------------+------------+----------+----------+

+--------+------------+--------------+----+---+--------+---------+--------+--------------------+---------+--------------+--------------+------------+------------+----------------------+-----------------+---------------+---------------+-----------------+----------------------+------------------+---------------+--------------+---------------------+-------------------+-------------------+----------------+-------------------------+-------------------------+----------------------------+---------------------------+------------------+
|station |station_code|data_formatada|ano |mes|latitude|longitude|altitude|macro_regiao_sp     |tipo_area|faixa_altitude|temp_media_dia|temp_min_dia|temp_max_dia|temp_orvalho_media_dia|umidade_media_dia|umidade_min_

## 21. Criação do dataset para prever a temperatura média dos próximos 7 dias

O alvo será:

`temperatura_media_proximos_7_dias`

Ele representa a média da temperatura média diária dos próximos 7 dias para a mesma estação.

In [63]:
base_diaria_features.createOrReplaceTempView("base_diaria_features")

dataset_semana = spark.sql("""
    SELECT
        *,
        AVG(temp_media_dia) OVER (
            PARTITION BY station_code
            ORDER BY data_formatada
            ROWS BETWEEN 1 FOLLOWING AND 7 FOLLOWING
        ) AS temperatura_media_proximos_7_dias
    FROM base_diaria_features
""")

dataset_semana.createOrReplaceTempView("dataset_semana_com_null")

dataset_semana = spark.sql("""
    SELECT *
    FROM dataset_semana_com_null
    WHERE temperatura_media_proximos_7_dias IS NOT NULL
""")

dataset_semana.createOrReplaceTempView("dataset_semana")

spark.sql("""
    SELECT
        COUNT(*) AS total_linhas,
        MIN(data_formatada) AS data_inicial,
        MAX(data_formatada) AS data_final,
        ROUND(AVG(temperatura_media_proximos_7_dias), 2) AS media_alvo
    FROM dataset_semana
""").show(truncate=False)

dataset_semana.show(10, truncate=False)

+------------+------------+----------+----------+
|total_linhas|data_inicial|data_final|media_alvo|
+------------+------------+----------+----------+
|166172      |2001-08-30  |2021-04-29|21.74     |
+------------+------------+----------+----------+

+--------+------------+--------------+----+---+--------+---------+--------+--------------------+---------+--------------+--------------+------------+------------+----------------------+-----------------+---------------+---------------+-----------------+----------------------+------------------+---------------+--------------+---------------------+-------------------+-------------------+----------------+-------------------------+-------------------------+----------------------------+---------------------------+---------------------------------+
|station |station_code|data_formatada|ano |mes|latitude|longitude|altitude|macro_regiao_sp     |tipo_area|faixa_altitude|temp_media_dia|temp_min_dia|temp_max_dia|temp_orvalho_media_dia|umidade_media_d

## 22. Preparação final do dataset para previsão de amanhã

Nesta etapa, o dataset com alvo `temperatura_amanha` é preparado para modelagem.

A base é separada temporalmente:

- anos anteriores a 2018 são usados para treino;
- anos de 2018 em diante são usados para teste.

As variáveis categóricas geográficas são transformadas em índices numéricos com `StringIndexer`, pois os modelos do Spark MLlib trabalham com features numéricas.

Os valores ausentes das variáveis numéricas são preenchidos com mediana usando `Imputer`.

Tanto o `StringIndexer` quanto o `Imputer` são ajustados apenas na base de treino e depois aplicados na base de teste, evitando vazamento de dados.

In [64]:
dataset_amanha.createOrReplaceTempView("dataset_amanha")

df_amanha_base = spark.sql("""
    SELECT
        station,
        station_code,
        data_formatada,
        ano,

        mes_sin,
        mes_cos,
        latitude,
        longitude,
        altitude,

        temp_media_dia,
        temp_min_dia,
        temp_max_dia,
        temp_orvalho_media_dia,

        umidade_media_dia,
        umidade_min_dia,
        umidade_max_dia,

        pressao_media_dia,
        precipitacao_total_dia,
        radiacao_media_dia,
        vento_medio_dia,
        rajada_max_dia,

        temp_media_ontem,
        temp_media_ultimos_3_dias,
        temp_media_ultimos_7_dias,
        umidade_media_ultimos_7_dias,
        precipitacao_ultimos_7_dias,

        COALESCE(macro_regiao_sp, 'sem_info') AS macro_regiao_sp,
        COALESCE(tipo_area, 'sem_info') AS tipo_area,
        COALESCE(faixa_altitude, 'sem_info') AS faixa_altitude,

        temperatura_amanha
    FROM dataset_amanha
    WHERE temperatura_amanha IS NOT NULL
      AND ano IS NOT NULL
""")

df_amanha_base.createOrReplaceTempView("df_amanha_base")

spark.sql("""
    SELECT
        COUNT(*) AS total_linhas,
        MIN(data_formatada) AS data_inicial,
        MAX(data_formatada) AS data_final,
        ROUND(AVG(temperatura_amanha), 2) AS media_alvo
    FROM df_amanha_base
""").show(truncate=False)

+------------+------------+----------+----------+
|total_linhas|data_inicial|data_final|media_alvo|
+------------+------------+----------+----------+
|166172      |2001-08-30  |2021-04-29|21.74     |
+------------+------------+----------+----------+



Separação temporal

In [65]:
df_amanha_train = spark.sql("""
    SELECT *
    FROM df_amanha_base
    WHERE ano < 2018
""")

df_amanha_test = spark.sql("""
    SELECT *
    FROM df_amanha_base
    WHERE ano >= 2018
""")

print(f"Treino amanhã: {df_amanha_train.count():,}")
print(f"Teste amanhã : {df_amanha_test.count():,}")

Treino amanhã: 120,232
Teste amanhã : 45,940


Aplicação de StringIndexer

In [66]:
indexer_macro_amanha = StringIndexer(
    inputCol="macro_regiao_sp",
    outputCol="macro_regiao_sp_idx",
    handleInvalid="keep"
)

indexer_tipo_amanha = StringIndexer(
    inputCol="tipo_area",
    outputCol="tipo_area_idx",
    handleInvalid="keep"
)

indexer_altitude_amanha = StringIndexer(
    inputCol="faixa_altitude",
    outputCol="faixa_altitude_idx",
    handleInvalid="keep"
)

modelo_macro_amanha = indexer_macro_amanha.fit(df_amanha_train)
modelo_tipo_amanha = indexer_tipo_amanha.fit(df_amanha_train)
modelo_altitude_amanha = indexer_altitude_amanha.fit(df_amanha_train)

df_amanha_train = modelo_macro_amanha.transform(df_amanha_train)
df_amanha_test = modelo_macro_amanha.transform(df_amanha_test)

df_amanha_train = modelo_tipo_amanha.transform(df_amanha_train)
df_amanha_test = modelo_tipo_amanha.transform(df_amanha_test)

df_amanha_train = modelo_altitude_amanha.transform(df_amanha_train)
df_amanha_test = modelo_altitude_amanha.transform(df_amanha_test)

Imputação de dados na base de treino

In [67]:
colunas_para_imputar = [
    "ano",
    "mes_sin",
    "mes_cos",
    "latitude",
    "longitude",
    "altitude",

    "temp_media_dia",
    "temp_min_dia",
    "temp_max_dia",
    "temp_orvalho_media_dia",

    "umidade_media_dia",
    "umidade_min_dia",
    "umidade_max_dia",

    "pressao_media_dia",
    "precipitacao_total_dia",
    "radiacao_media_dia",
    "vento_medio_dia",
    "rajada_max_dia",

    "temp_media_ontem",
    "temp_media_ultimos_3_dias",
    "temp_media_ultimos_7_dias",
    "umidade_media_ultimos_7_dias",
    "precipitacao_ultimos_7_dias"
]

imputer_amanha = Imputer(
    inputCols=colunas_para_imputar,
    outputCols=[c + "_imputado" for c in colunas_para_imputar]
).setStrategy("median")

modelo_imputer_amanha = imputer_amanha.fit(df_amanha_train)

df_amanha_train = modelo_imputer_amanha.transform(df_amanha_train)
df_amanha_test = modelo_imputer_amanha.transform(df_amanha_test)

Montar a base final

In [68]:
features_numericas_imputadas = [
    c + "_imputado"
    for c in colunas_para_imputar
]

features_categoricas_indexadas = [
    "macro_regiao_sp_idx",
    "tipo_area_idx",
    "faixa_altitude_idx"
]

features_amanha = features_numericas_imputadas + features_categoricas_indexadas

colunas_finais_amanha = [
    "station",
    "station_code",
    "data_formatada"
] + features_amanha + ["temperatura_amanha"]

dataset_amanha_train = df_amanha_train.select(*colunas_finais_amanha)
dataset_amanha_test = df_amanha_test.select(*colunas_finais_amanha)

dataset_amanha_final = dataset_amanha_train.unionByName(dataset_amanha_test)

print(f"Treino final amanhã: {dataset_amanha_train.count():,}")
print(f"Teste final amanhã : {dataset_amanha_test.count():,}")
print(f"Completo amanhã    : {dataset_amanha_final.count():,}")

Treino final amanhã: 120,232
Teste final amanhã : 45,940
Completo amanhã    : 166,172


## 23. Preparação final do dataset para previsão dos próximos 7 dias

Nesta etapa, o dataset com alvo `temperatura_media_proximos_7_dias` é preparado para modelagem.

O processo é equivalente ao dataset de amanhã:

- seleção das mesmas features explicativas;
- separação temporal entre treino e teste;
- indexação das variáveis categóricas;
- imputação das variáveis numéricas com mediana;
- criação das bases finais de treino, teste e dataset completo.

A diferença está no alvo: aqui o modelo tentará prever a temperatura média dos próximos 7 dias para cada estação meteorológica.

In [69]:
dataset_semana.createOrReplaceTempView("dataset_semana")

df_semana_base = spark.sql("""
    SELECT
        station,
        station_code,
        data_formatada,
        ano,

        mes_sin,
        mes_cos,
        latitude,
        longitude,
        altitude,

        temp_media_dia,
        temp_min_dia,
        temp_max_dia,
        temp_orvalho_media_dia,

        umidade_media_dia,
        umidade_min_dia,
        umidade_max_dia,

        pressao_media_dia,
        precipitacao_total_dia,
        radiacao_media_dia,
        vento_medio_dia,
        rajada_max_dia,

        temp_media_ontem,
        temp_media_ultimos_3_dias,
        temp_media_ultimos_7_dias,
        umidade_media_ultimos_7_dias,
        precipitacao_ultimos_7_dias,

        COALESCE(macro_regiao_sp, 'sem_info') AS macro_regiao_sp,
        COALESCE(tipo_area, 'sem_info') AS tipo_area,
        COALESCE(faixa_altitude, 'sem_info') AS faixa_altitude,

        temperatura_media_proximos_7_dias
    FROM dataset_semana
    WHERE temperatura_media_proximos_7_dias IS NOT NULL
      AND ano IS NOT NULL
""")

df_semana_base.createOrReplaceTempView("df_semana_base")

spark.sql("""
    SELECT
        COUNT(*) AS total_linhas,
        MIN(data_formatada) AS data_inicial,
        MAX(data_formatada) AS data_final,
        ROUND(AVG(temperatura_media_proximos_7_dias), 2) AS media_alvo
    FROM df_semana_base
""").show(truncate=False)

+------------+------------+----------+----------+
|total_linhas|data_inicial|data_final|media_alvo|
+------------+------------+----------+----------+
|166172      |2001-08-30  |2021-04-29|21.74     |
+------------+------------+----------+----------+



Separação temporal

In [70]:
df_semana_train = spark.sql("""
    SELECT *
    FROM df_semana_base
    WHERE ano < 2018
""")

df_semana_test = spark.sql("""
    SELECT *
    FROM df_semana_base
    WHERE ano >= 2018
""")

print(f"Treino semana: {df_semana_train.count():,}")
print(f"Teste semana : {df_semana_test.count():,}")

Treino semana: 120,232
Teste semana : 45,940


Indexação das categorias

In [71]:
indexer_macro_semana = StringIndexer(
    inputCol="macro_regiao_sp",
    outputCol="macro_regiao_sp_idx",
    handleInvalid="keep"
)

indexer_tipo_semana = StringIndexer(
    inputCol="tipo_area",
    outputCol="tipo_area_idx",
    handleInvalid="keep"
)

indexer_altitude_semana = StringIndexer(
    inputCol="faixa_altitude",
    outputCol="faixa_altitude_idx",
    handleInvalid="keep"
)

modelo_macro_semana = indexer_macro_semana.fit(df_semana_train)
modelo_tipo_semana = indexer_tipo_semana.fit(df_semana_train)
modelo_altitude_semana = indexer_altitude_semana.fit(df_semana_train)

df_semana_train = modelo_macro_semana.transform(df_semana_train)
df_semana_test = modelo_macro_semana.transform(df_semana_test)

df_semana_train = modelo_tipo_semana.transform(df_semana_train)
df_semana_test = modelo_tipo_semana.transform(df_semana_test)

df_semana_train = modelo_altitude_semana.transform(df_semana_train)
df_semana_test = modelo_altitude_semana.transform(df_semana_test)

Imputação

In [72]:
imputer_semana = Imputer(
    inputCols=colunas_para_imputar,
    outputCols=[c + "_imputado" for c in colunas_para_imputar]
).setStrategy("median")

modelo_imputer_semana = imputer_semana.fit(df_semana_train)

df_semana_train = modelo_imputer_semana.transform(df_semana_train)
df_semana_test = modelo_imputer_semana.transform(df_semana_test)

Montagem final

In [73]:
features_numericas_imputadas_semana = [
    c + "_imputado"
    for c in colunas_para_imputar
]

features_categoricas_indexadas_semana = [
    "macro_regiao_sp_idx",
    "tipo_area_idx",
    "faixa_altitude_idx"
]

features_semana = features_numericas_imputadas_semana + features_categoricas_indexadas_semana

colunas_finais_semana = [
    "station",
    "station_code",
    "data_formatada"
] + features_semana + ["temperatura_media_proximos_7_dias"]

dataset_semana_train = df_semana_train.select(*colunas_finais_semana)
dataset_semana_test = df_semana_test.select(*colunas_finais_semana)

dataset_semana_final = dataset_semana_train.unionByName(dataset_semana_test)

print(f"Treino final semana: {dataset_semana_train.count():,}")
print(f"Teste final semana : {dataset_semana_test.count():,}")
print(f"Completo semana    : {dataset_semana_final.count():,}")

Treino final semana: 120,232
Teste final semana : 45,940
Completo semana    : 166,172


## 24. Verificação de nulos nos datasets finais

Após a imputação, os datasets finais devem estar sem valores nulos nas principais features e nos alvos.

Esta verificação confirma se o processo de indexação das variáveis categóricas e imputação das variáveis numéricas funcionou corretamente antes do salvamento das bases para modelagem.

Verificação de nulos de amanhã

In [74]:
dataset_amanha_final.createOrReplaceTempView("dataset_amanha_final")

spark.sql("""
    SELECT
        SUM(CASE WHEN temperatura_amanha IS NULL THEN 1 ELSE 0 END) AS nulos_alvo,
        SUM(CASE WHEN temp_media_dia_imputado IS NULL THEN 1 ELSE 0 END) AS nulos_temp_media,
        SUM(CASE WHEN umidade_media_dia_imputado IS NULL THEN 1 ELSE 0 END) AS nulos_umidade,
        SUM(CASE WHEN pressao_media_dia_imputado IS NULL THEN 1 ELSE 0 END) AS nulos_pressao,
        SUM(CASE WHEN precipitacao_total_dia_imputado IS NULL THEN 1 ELSE 0 END) AS nulos_precipitacao,
        SUM(CASE WHEN macro_regiao_sp_idx IS NULL THEN 1 ELSE 0 END) AS nulos_macro_regiao,
        SUM(CASE WHEN tipo_area_idx IS NULL THEN 1 ELSE 0 END) AS nulos_tipo_area,
        SUM(CASE WHEN faixa_altitude_idx IS NULL THEN 1 ELSE 0 END) AS nulos_faixa_altitude
    FROM dataset_amanha_final
""").show(truncate=False)

+----------+----------------+-------------+-------------+------------------+------------------+---------------+--------------------+
|nulos_alvo|nulos_temp_media|nulos_umidade|nulos_pressao|nulos_precipitacao|nulos_macro_regiao|nulos_tipo_area|nulos_faixa_altitude|
+----------+----------------+-------------+-------------+------------------+------------------+---------------+--------------------+
|0         |0               |0            |0            |0                 |0                 |0              |0                   |
+----------+----------------+-------------+-------------+------------------+------------------+---------------+--------------------+



Verificação de nulos dos próximos 7 dias

In [75]:
dataset_semana_final.createOrReplaceTempView("dataset_semana_final")

spark.sql("""
    SELECT
        SUM(CASE WHEN temperatura_media_proximos_7_dias IS NULL THEN 1 ELSE 0 END) AS nulos_alvo,
        SUM(CASE WHEN temp_media_dia_imputado IS NULL THEN 1 ELSE 0 END) AS nulos_temp_media,
        SUM(CASE WHEN umidade_media_dia_imputado IS NULL THEN 1 ELSE 0 END) AS nulos_umidade,
        SUM(CASE WHEN pressao_media_dia_imputado IS NULL THEN 1 ELSE 0 END) AS nulos_pressao,
        SUM(CASE WHEN precipitacao_total_dia_imputado IS NULL THEN 1 ELSE 0 END) AS nulos_precipitacao,
        SUM(CASE WHEN macro_regiao_sp_idx IS NULL THEN 1 ELSE 0 END) AS nulos_macro_regiao,
        SUM(CASE WHEN tipo_area_idx IS NULL THEN 1 ELSE 0 END) AS nulos_tipo_area,
        SUM(CASE WHEN faixa_altitude_idx IS NULL THEN 1 ELSE 0 END) AS nulos_faixa_altitude
    FROM dataset_semana_final
""").show(truncate=False)

+----------+----------------+-------------+-------------+------------------+------------------+---------------+--------------------+
|nulos_alvo|nulos_temp_media|nulos_umidade|nulos_pressao|nulos_precipitacao|nulos_macro_regiao|nulos_tipo_area|nulos_faixa_altitude|
+----------+----------------+-------------+-------------+------------------+------------------+---------------+--------------------+
|0         |0               |0            |0            |0                 |0                 |0              |0                   |
+----------+----------------+-------------+-------------+------------------+------------------+---------------+--------------------+



## 25. Estatísticas finais dos alvos

Nesta etapa são calculadas estatísticas descritivas dos dois alvos criados:

- `temperatura_amanha`;
- `temperatura_media_proximos_7_dias`.

Essas estatísticas ajudam a validar se os alvos possuem valores coerentes antes da etapa de modelagem.

Como o objetivo do projeto é regressão, é importante verificar média, mínimo, máximo e desvio padrão das variáveis que serão previstas.

Estatísticas amanhã

In [76]:
dataset_amanha_final.createOrReplaceTempView("dataset_amanha_final")

spark.sql("""
    SELECT
        COUNT(*) AS total_registros,
        ROUND(AVG(temperatura_amanha), 2) AS media_temperatura_amanha,
        ROUND(MIN(temperatura_amanha), 2) AS min_temperatura_amanha,
        ROUND(MAX(temperatura_amanha), 2) AS max_temperatura_amanha,
        ROUND(STDDEV(temperatura_amanha), 2) AS desvio_temperatura_amanha
    FROM dataset_amanha_final
""").show(truncate=False)

+---------------+------------------------+----------------------+----------------------+-------------------------+
|total_registros|media_temperatura_amanha|min_temperatura_amanha|max_temperatura_amanha|desvio_temperatura_amanha|
+---------------+------------------------+----------------------+----------------------+-------------------------+
|166172         |21.74                   |-9.74                 |39.33                 |3.83                     |
+---------------+------------------------+----------------------+----------------------+-------------------------+



Estatísticas próximos 7 dias

In [77]:
dataset_semana_final.createOrReplaceTempView("dataset_semana_final")

spark.sql("""
    SELECT
        COUNT(*) AS total_registros,
        ROUND(AVG(temperatura_media_proximos_7_dias), 2) AS media_temperatura_semana,
        ROUND(MIN(temperatura_media_proximos_7_dias), 2) AS min_temperatura_semana,
        ROUND(MAX(temperatura_media_proximos_7_dias), 2) AS max_temperatura_semana,
        ROUND(STDDEV(temperatura_media_proximos_7_dias), 2) AS desvio_temperatura_semana
    FROM dataset_semana_final
""").show(truncate=False)

+---------------+------------------------+----------------------+----------------------+-------------------------+
|total_registros|media_temperatura_semana|min_temperatura_semana|max_temperatura_semana|desvio_temperatura_semana|
+---------------+------------------------+----------------------+----------------------+-------------------------+
|166172         |21.74                   |-9.74                 |38.12                 |3.45                     |
+---------------+------------------------+----------------------+----------------------+-------------------------+



## 26. Salvamento das bases finais

Nesta etapa, as bases finais são salvas em formato Parquet para serem usadas diretamente nos notebooks de modelagem.

Serão salvas:

- base diária agregada;
- dataset completo para previsão de amanhã;
- treino e teste para previsão de amanhã;
- dataset completo para previsão dos próximos 7 dias;
- treino e teste para previsão dos próximos 7 dias.

O formato Parquet é mantido por ser mais eficiente para leitura, compressão e reutilização em projetos de Big Data.

In [78]:
base_diaria_path = "/home/jovyan/work/data/processed/weather_sp_base_diaria"

dataset_amanha_path = "/home/jovyan/work/data/processed/weather_sp_dataset_amanha"
dataset_amanha_train_path = "/home/jovyan/work/data/processed/weather_sp_amanha_train"
dataset_amanha_test_path = "/home/jovyan/work/data/processed/weather_sp_amanha_test"

dataset_semana_path = "/home/jovyan/work/data/processed/weather_sp_dataset_semana"
dataset_semana_train_path = "/home/jovyan/work/data/processed/weather_sp_semana_train"
dataset_semana_test_path = "/home/jovyan/work/data/processed/weather_sp_semana_test"

In [79]:
base_diaria = base_diaria.coalesce(4)

dataset_amanha_final = dataset_amanha_final.coalesce(4)
dataset_amanha_train = dataset_amanha_train.coalesce(4)
dataset_amanha_test = dataset_amanha_test.coalesce(4)

dataset_semana_final = dataset_semana_final.coalesce(4)
dataset_semana_train = dataset_semana_train.coalesce(4)
dataset_semana_test = dataset_semana_test.coalesce(4)

In [80]:
base_diaria.write.mode("overwrite").parquet(base_diaria_path)

dataset_amanha_final.write.mode("overwrite").parquet(dataset_amanha_path)
dataset_amanha_train.write.mode("overwrite").parquet(dataset_amanha_train_path)
dataset_amanha_test.write.mode("overwrite").parquet(dataset_amanha_test_path)

dataset_semana_final.write.mode("overwrite").parquet(dataset_semana_path)
dataset_semana_train.write.mode("overwrite").parquet(dataset_semana_train_path)
dataset_semana_test.write.mode("overwrite").parquet(dataset_semana_test_path)

print(f"Base diária salva em: {base_diaria_path}")

print(f"Dataset amanhã completo salvo em: {dataset_amanha_path}")
print(f"Dataset amanhã treino salvo em   : {dataset_amanha_train_path}")
print(f"Dataset amanhã teste salvo em    : {dataset_amanha_test_path}")

print(f"Dataset semana completo salvo em: {dataset_semana_path}")
print(f"Dataset semana treino salvo em   : {dataset_semana_train_path}")
print(f"Dataset semana teste salvo em    : {dataset_semana_test_path}")

Base diária salva em: /home/jovyan/work/data/processed/weather_sp_base_diaria
Dataset amanhã completo salvo em: /home/jovyan/work/data/processed/weather_sp_dataset_amanha
Dataset amanhã treino salvo em   : /home/jovyan/work/data/processed/weather_sp_amanha_train
Dataset amanhã teste salvo em    : /home/jovyan/work/data/processed/weather_sp_amanha_test
Dataset semana completo salvo em: /home/jovyan/work/data/processed/weather_sp_dataset_semana
Dataset semana treino salvo em   : /home/jovyan/work/data/processed/weather_sp_semana_train
Dataset semana teste salvo em    : /home/jovyan/work/data/processed/weather_sp_semana_test


## Conclusão do pré-processamento

Neste notebook, a base meteorológica de São Paulo foi preparada para modelagem preditiva.

Principais entregas:

- base horária limpa;
- base diária agregada por estação;
- dataset para prever a temperatura média de amanhã;
- dataset para prever a temperatura média dos próximos 7 dias;
- separação temporal entre treino e teste;
- imputação sem vazamento de dados;
- indexação de variáveis categóricas geográficas;
- salvamento das bases em Parquet.

Os próximos notebooks podem usar diretamente:

- `weather_sp_amanha_train`;
- `weather_sp_amanha_test`;
- `weather_sp_semana_train`;
- `weather_sp_semana_test`.

Essas bases serão usadas para treinar e comparar:

- Linear Regression;
- Random Forest Regressor;
- Redes Neurais.